# Finding most recent sequences from full dataset

* California Cow (B3.13) most recent sequence
* Idaho Cow (B3.13) most recent sequence
* Nevada Cow (D1.1) most recent sequence

In [21]:
import os
import pandas as pd
import dateutil

# home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
dataset_new_d1_1 = home + "Combinations/GISAID_Andersen_NCBI_Virus/04-14-2025--06-05-2025_D1_1/"
dataset_new_b3_13 = home + "Combinations/GISAID_Andersen_NCBI_Virus/04-14-2025--06-05-2025_B3_13/"
dataset_old = home + "Combinations/GISAID_Andersen_NCBI_Virus/11-01-2021--04-14-2025/"
# states = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu/references/"
states = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu/references/"

os.chdir(states)
states_ref = pd.read_csv("states_ref.csv")

In [37]:
# Function to prepare dataframes
def fasta_df_new(file_name, state_ref):

    fasta = pd.DataFrame()
    headers = []
    isolate_ids = []
    isolate_names = []
    subtypes = []
    segments = []
    collection_dates = []
    sequences = []
    host_types = []
    species = []
    genotypes = []
    with open(file_name) as f:
        lines = f.readlines()
        for num, line in enumerate(lines):
            # print(line)
            if line[0] == ">": # If it's a header
                if line[1:].strip() not in headers: # And the previous line is not a header we've seen before
                    header = line[1:].strip() # Remove the ">"
                    # print(header)
                    split_header = header.split("|")
                    split_first_header = split_header[1].split("/")
                    # print(split_header)
                    headers.append(header) 
                    isolate_ids.append(split_first_header[3])
                    isolate_names.append(split_header[1]) # We'll need to extract data from this too
                    # print(split_header[2].split("_")[-1])
                    subtypes.append(split_header[2])  # Get only H5N1
                    genotypes.append(split_header[-1])
                    segments.append(file_name.split("_")[-3])
                    host_types.append(split_header[-2])
                    species.append(split_first_header[1])
                    # if split_header[4] == "2024-01-01":
                    #     collection_dates.append("2024") # No samples were collected 1/1/2024, these are all unknown 
                    # elif split_header[4] == "2025-01-01":
                    #     collection_dates.append("2025")
                    # else: 
                    collection_dates.append(split_header[4].split("_")[-1])
                    if num < len(lines): # If we're not at the last line
                        # for i, l in enumerate(lines[num + 1:]):
                        i = num
                        sequence = ""
                        # print(lines[i])
                        # print(lines[i + 1])
                        while i < len(lines) - 1 and lines[i + 1][0] != ">": # While the next line is part of a sequence
                            sequence = sequence + lines[i + 1].strip()
                            i += 1
                        sequences.append(sequence) # Add next line to sequences
        f.close()

    # Create columns for data frame 
    fasta["Header"] = headers
    fasta["Isolate_Id"] = isolate_ids
    fasta["Isolate_Name"] = isolate_names
    fasta["Subtype"] = subtypes
    fasta["Segment"] = segments
    # Geo_Location is more complicated
    fasta["Geo_Location"] = fasta["Header"].apply(lambda x: state_ref.loc[state_ref["Abbreviation"] == x.split("/")[2], 'Country'].iloc[0] + "-" + x.split("/")[2] if x.split("/")[2] in state_ref["Abbreviation"].values else state_ref.loc[state_ref["State"] == x.split("/")[2].replace("_", " "), 'Country'].iloc[0] + "-" + state_ref.loc[state_ref["State"] == x.split("/")[2].replace("_", " "), 'Abbreviation'].iloc[0] if x.split("/")[2].replace("_", " ") in state_ref["State"].values else x.split("/")[2].replace(": ", "-"))
    fasta["Date Collected"] = collection_dates
    fasta["Species"] = species
    fasta["Host_Type"] = host_types
    fasta["Genotype"] = genotypes
    fasta["Sequence"] = sequences
    
    return fasta

# Function to prepare dataframes
def fasta_df_old(file_name, state_ref):

    fasta = pd.DataFrame()
    headers = []
    isolate_ids = []
    isolate_names = []
    subtypes = []
    segments = []
    collection_dates = []
    sequences = []
    host_types = []
    species = []
    genotypes = []
    with open(file_name) as f:
        lines = f.readlines()
        for num, line in enumerate(lines):
            # print(line)
            if line[0] == ">": # If it's a header
                if line[1:].strip() not in headers: # And the previous line is not a header we've seen before
                    header = line[1:].strip() # Remove the ">"
                    # print(header)
                    split_header = header.split("|")
                    split_first_header = split_header[0].split("/")
                    # print(split_header)
                    headers.append(header) 
                    isolate_ids.append(split_first_header[3])
                    isolate_names.append(split_header[0]) # We'll need to extract data from this too
                    # print(split_header[2].split("_")[-1])
                    subtypes.append(split_header[1])  # Get only H5N1
                    genotypes.append(split_header[-1])
                    segments.append(file_name.split("_")[-3])
                    host_types.append(split_header[-2])
                    species.append(split_first_header[1])
                    # if split_header[4] == "2024-01-01":
                    #     collection_dates.append("2024") # No samples were collected 1/1/2024, these are all unknown 
                    # elif split_header[4] == "2025-01-01":
                    #     collection_dates.append("2025")
                    # else: 
                    collection_dates.append(split_header[2].split("_")[-1])
                    if num < len(lines): # If we're not at the last line
                        # for i, l in enumerate(lines[num + 1:]):
                        i = num
                        sequence = ""
                        # print(lines[i])
                        # print(lines[i + 1])
                        while i < len(lines) - 1 and lines[i + 1][0] != ">": # While the next line is part of a sequence
                            sequence = sequence + lines[i + 1].strip()
                            i += 1
                        sequences.append(sequence) # Add next line to sequences
        f.close()

    # Create columns for data frame 
    fasta["Header"] = headers
    fasta["Isolate_Id"] = isolate_ids
    fasta["Isolate_Name"] = isolate_names
    fasta["Subtype"] = subtypes
    fasta["Segment"] = segments
    # Geo_Location is more complicated
    fasta["Geo_Location"] = fasta["Header"].apply(lambda x: state_ref.loc[state_ref["Abbreviation"] == x.split("/")[2], 'Country'].iloc[0] + "-" + x.split("/")[2] if x.split("/")[2] in state_ref["Abbreviation"].values else state_ref.loc[state_ref["State"] == x.split("/")[2].replace("_", " "), 'Country'].iloc[0] + "-" + state_ref.loc[state_ref["State"] == x.split("/")[2].replace("_", " "), 'Abbreviation'].iloc[0] if x.split("/")[2].replace("_", " ") in state_ref["State"].values else x.split("/")[2].replace(": ", "-"))
    fasta["Date Collected"] = collection_dates
    fasta["Species"] = species
    fasta["Host_Type"] = host_types
    fasta["Genotype"] = genotypes
    fasta["Sequence"] = sequences
    
    return fasta



In [45]:
os.chdir(dataset_new_b3_13)

new_b3_13_ha = fasta_df_new("B3.13_HA_combined_04-14-2025--06-05-2025.fasta", states_ref)
new_b3_13_ha_cattle_ca = new_b3_13_ha[(new_b3_13_ha["Host_Type"] == "cattle") & (new_b3_13_ha["Geo_Location"] == "USA-CA")]
new_b3_13_ha_cattle_id = new_b3_13_ha[(new_b3_13_ha["Host_Type"] == "cattle") & (new_b3_13_ha["Geo_Location"] == "USA-ID")]

os.chdir(dataset_old) 
print(os.listdir())

old_b3_13_ha = fasta_df_old("B3.13_HA_combined_04-14-2025.fasta", states_ref)
old_b3_13_ha_cattle_id = old_b3_13_ha[(old_b3_13_ha["Host_Type"] == "cattle") & (old_b3_13_ha["Geo_Location"] == "USA-ID")]
# old_b3_13_ha_cattle_id["True_Date"] = old_b3_13_ha_cattle_id["Header"].apply(lambda x: x.split("|")[2])

print(new_b3_13_ha_cattle_ca.sort_values(by="Date Collected"))
print(new_b3_13_ha_cattle_id.sort_values(by="Date Collected"))

old_d1_1_ha = fasta_df_old("D1.1_HA_combined_04-14-2025.fasta", states_ref)
old_d1_1_ha_cattle_nv = old_d1_1_ha[(old_d1_1_ha["Host_Type"] == "cattle") & (old_d1_1_ha["Geo_Location"] == "USA-NV")]
# old_d1_1_ha_cattle_nv["True_Date"] = old_d1_1_ha_cattle_nv["Header"].apply(lambda x: x.split("|")[2])

print(old_d1_1_ha_cattle_nv.sort_values(by="Date Collected"))

os.chdir(dataset_new_d1_1)
new_d1_1_ha = fasta_df_new("D1.1_HA_combined_04-14-2025--06-05-2025.fasta", states_ref)
new_d1_1_ha_cattle_nv = new_d1_1_ha[(new_d1_1_ha["Host_Type"] == "cattle") & (new_d1_1_ha["Geo_Location"] == "USA-NV")]
# new_d1_1_ha_cattle_nv["True_Date"] = new_d1_1_ha_cattle_nv["Header"].apply(lambda x: x.split("|")[2])


print(new_d1_1_ha_cattle_nv.sort_values(by="Date Collected"))

# new_b3_13_ha["Geo_Location"] = new_b3_13_ha["full_header"].apply(lambda x: x.split("|")[2])
# new_b3_13_ha["Date"] = new_b3_13_ha["full_header"].apply(lambda x: x.split("|")[3])
# new_b3_13_ha["Host"] = new_b3_13_ha["full_header"].apply(lambda x: x.split("|")[-2])

# print(new_b3_13_ha)

['A2_HA_combined_04-14-2025.fasta', 'A2_MP_combined_04-14-2025.fasta', 'A2_NA_combined_04-14-2025.fasta', 'A2_NP_combined_04-14-2025.fasta', 'A2_NS_combined_04-14-2025.fasta', 'A2_PA_combined_04-14-2025.fasta', 'A2_PB1_combined_04-14-2025.fasta', 'A2_PB2_combined_04-14-2025.fasta', 'A3_HA_combined_04-14-2025.fasta', 'A3_MP_combined_04-14-2025.fasta', 'A3_NA_combined_04-14-2025.fasta', 'A3_NP_combined_04-14-2025.fasta', 'A3_NS_combined_04-14-2025.fasta', 'A3_PA_combined_04-14-2025.fasta', 'A3_PB1_combined_04-14-2025.fasta', 'A3_PB2_combined_04-14-2025.fasta', 'B3.13_HA_combined_04-14-2025.fasta', 'B3.13_MP_combined_04-14-2025.fasta', 'B3.13_NA_combined_04-14-2025.fasta', 'B3.13_NP_combined_04-14-2025.fasta', 'B3.13_NS_combined_04-14-2025.fasta', 'B3.13_PA_combined_04-14-2025.fasta', 'B3.13_PB1_combined_04-14-2025.fasta', 'B3.13_PB2_combined_04-14-2025.fasta', 'B3.2_HA_combined_04-14-2025.fasta', 'B3.2_MP_combined_04-14-2025.fasta', 'B3.2_NA_combined_04-14-2025.fasta', 'B3.2_NP_combined_

In [46]:
os.chdir(home + "Other/")

usda_data = pd.read_csv("Table Details by Date_Full Data_data.csv")

usda_data["Confirmed"] = usda_data["Confirmed"].apply(lambda x: dateutil.parser.parse(x))

usda_data = usda_data.sort_values(by=["State", "Confirmed"])

print(usda_data[usda_data["State"] == "California"])
print(usda_data[usda_data["State"] == "Idaho"])
print(usda_data[usda_data["State"] == "Nevada"])

     Confirmed       State Special Id          Production Species  \
853 2024-08-30  California     CA 002  Dairy Milking Cows  Cattle   
854 2024-08-30  California     CA 003  Dairy Milking Cows  Cattle   
855 2024-08-30  California     CA 001  Dairy Milking Cows  Cattle   
850 2024-09-09  California     CA 006  Dairy Milking Cows  Cattle   
851 2024-09-09  California     CA 005  Dairy Milking Cows  Cattle   
..         ...         ...        ...                 ...     ...   
136 2025-04-11  California     CA 741  Dairy Milking Cows  Cattle   
90  2025-04-14  California     CA 756  Dairy Milking Cows  Cattle   
78  2025-04-21  California     CA 772  Dairy Milking Cows  Cattle   
86  2025-04-30  California     CA 764  Dairy Milking Cows  Cattle   
0   2025-06-03  California     CA 773  Dairy Milking Cows  Cattle   

     Search filter  Table Sort Filter    Confirmed Diagnosis  \
853           True  Confirmation Date  8/30/2024 12:00:00 AM   
854           True  Confirmation Date  8/30

In [48]:
old_b3_13_ha[(old_b3_13_ha["Host_Type"] == "avian") & (old_b3_13_ha["Date Collected"].apply(lambda x: dateutil.parser.parse(x)) > dateutil.parser.parse("2025-01-01"))].sort_values(by="Date Collected", ascending=False) 
# old_b3_13_ha

,Header,Isolate_Id,Isolate_Name,Subtype,Segment,Geo_Location,Date Collected,Species,Host_Type,Genotype,Sequence
984,A/turkey/California/003903-001/2025|H5N1|2025-...,003903-001,A/turkey/California/003903-001/2025,H5N1,HA,USA-CA,2025-01-31,turkey,avian,B3.13,atgaagaacatagtactacttcttgcaatagttagccttgttaaaa...
3608,A/meleagris gallopavo/USA: CA/25-003903-001-or...,25-003903-001-original,A/meleagris gallopavo/USA: CA/25-003903-001-or...,H5N1,HA,USA-CA,2025-01-31,meleagris gallopavo,avian,B3.13,ATGAAGAACATAGTACTACTTCTTGCAATAGTTAGCCTTGTTAAAA...
3606,A/meleagris gallopavo/USA: CA/25-003574-002-or...,25-003574-002-original,A/meleagris gallopavo/USA: CA/25-003574-002-or...,H5N1,HA,USA-CA,2025-01-29,meleagris gallopavo,avian,B3.13,ATGAAGAACATAGTACTACTTCTTGCAATAGTTAGCCTTGTTAAAA...
982,A/turkey/California/003574-003/2025|H5N1|2025-...,003574-003,A/turkey/California/003574-003/2025,H5N1,HA,USA-CA,2025-01-29,turkey,avian,B3.13,atgaagaacatagtactacttcttgcaatagttagccttgttaaaa...
985,A/turkey/California/003574-001/2025|H5N1|2025-...,003574-001,A/turkey/California/003574-001/2025,H5N1,HA,USA-CA,2025-01-29,turkey,avian,B3.13,atgaagaacatagtactacttcttgcaatagttagccttgttaaaa...
987,A/turkey/California/003574-002/2025|H5N1|2025-...,003574-002,A/turkey/California/003574-002/2025,H5N1,HA,USA-CA,2025-01-29,turkey,avian,B3.13,atgaagaacatagtactacttcttgcaatagttagccttgttaaaa...
3607,A/meleagris gallopavo/USA: CA/25-003574-003-or...,25-003574-003-original,A/meleagris gallopavo/USA: CA/25-003574-003-or...,H5N1,HA,USA-CA,2025-01-29,meleagris gallopavo,avian,B3.13,ATGAAGAACATAGTACTACTTCTTGCAATAGTTAGCCTTGTTAAAA...
3605,A/meleagris gallopavo/USA: CA/25-003574-001-or...,25-003574-001-original,A/meleagris gallopavo/USA: CA/25-003574-001-or...,H5N1,HA,USA-CA,2025-01-29,meleagris gallopavo,avian,B3.13,ATGAAGAACATAGTACTACTTCTTGCAATAGTTAGCCTTGTTAAAA...
3503,A/anatidae/USA: CA/25-003063-003-original/2025...,25-003063-003-original,A/anatidae/USA: CA/25-003063-003-original/2025,H5N1,HA,USA-CA,2025-01-22,anatidae,avian,B3.13,ATGAAGAACATAGTACTACTTCTTGCAATAGTTAGCCTTGTTAAAA...
5638,A/gallus gallus/USA: CA/25-000136-003-original...,25-000136-003-original,A/gallus gallus/USA: CA/25-000136-003-original...,H5N1,HA,USA-CA,2025-01-02,gallus gallus,avian,B3.13,ATGAAGAACATAGTACTAATTCTTGCAATAGTTAGCCTTGTTAAAA...
